> **Disclaimer.** This project is educational and is not investment advice. Backtest results are historical and do not predict future results.

# *Beat the Market* on the WIN mini future: the article, redone

A section-by-section rebuild of the tables, charts and figures of Zarattini, Aziz and Barbon, ***Beat the Market: An Effective Intraday Momentum Strategy for S&P500 ETF (SPY)*** (version of Sept 22, 2025): sections 2 to 5, the appendix table and the FAQ. The article is on SPY, 2007 to 2024. This notebook runs the same rules on the **WIN mini Ibovespa future**, development period only (2021-12-01 to 2024-12-30, 770 days). Table and figure numbers follow the article.

**Read this before any number**
- **Development data only.** The loader returns bars before 2025-01-01 and the notebook stops if a later bar appears. The final test period has not been read. The owner decides when it runs.
- **Three years against seventeen.** With about 750 trading days a Sharpe ratio has a standard error near 0.6. Most differences between rows of the tables below are noise. The article's own tests ran on 2,600 days.
- **Every table is a look at the development data.** The brief counts them. Nothing here is chosen from, and the article's own warning applies: many slices, one sample.
- **Rules the article states but the authors' code does not implement** (the opposite-band stop of the base model, the current-band stop, the VWAP-only stop of Q22) are **my reading of the text** (`docs/brief.md` section 10). The final model (current band and VWAP, volatility-targeted size) is the authors' notebook, checked against the notebook's own loop.
- **Replaced inputs.** SPY buy and hold becomes the **IBOV** (a price index, so dividends are missing and it is understated). The **VIX** is not available: a 20-day realised volatility of the IBOV, from the days before, stands in for it and is labelled *proxy* wherever it appears.
- **Costs.** R$ 0.25 per contract and side (the owner's B3 round trip of R$ 0.50), no slippage unless a section says so. Positions are closed at the market close, as in the article.
- **Not recreated:** Q1 to Q3, Q9 to Q11, Q16, Q17 and Q25 (code, data vendors, TradingView, advice: not about numbers), Q15 (the I-Star slippage model is calibrated on US stocks), and Q24 (the performance after publication is, for us, the final test).

In [ ]:
import math
import warnings
from dataclasses import replace

import numpy as np
import pandas as pd
import plotly.graph_objects as go
from IPython.display import display
from plotly.subplots import make_subplots

from b3core.indicators import rsi
from b3core.intraday_momentum import (
    IntradayConfig,
    noise_bands,
    performance,
    run_intraday_momentum,
    session_vwap,
    strategy_exposure,
    trade_list,
)
from b3core.noise_area import noise_sigma
from b3core.winfut import FINAL_START, load_bars

pd.options.display.float_format = "{:,.2f}".format
pd.options.display.width = 220
pd.options.display.max_columns = 40
CAPITAL = 100_000.0

bars = load_bars("../samples")  # development period only
assert bars.index.max() < pd.Timestamp(FINAL_START.date()), "a bar from the final test period was loaded"
day = bars.index.normalize()
ibov = pd.read_csv("../samples/ifr2_universe/ibov.csv", index_col=0, parse_dates=True)["Close"].dropna()
ibov_ret = ibov.pct_change()
# 20-day realised volatility of the IBOV, annualised in %, from the days before (the VIX proxy)
rv20 = (ibov_ret.rolling(20).std() * math.sqrt(252) * 100).shift(1)

# the article closes every position at the market close: no exit time, no margin cap
BASE = IntradayConfig(exit_minute=24 * 60, margin_per_contract=0.0)
_runs: dict = {}


def run(**changes):
    cfg = replace(BASE, **changes)
    key = repr(cfg)
    if key not in _runs:
        _runs[key] = (cfg, run_intraday_momentum(bars, cfg))
    return _runs[key]


def ols(y, x):
    X = np.column_stack([np.ones(len(x)), x])
    beta, *_ = np.linalg.lstsq(X, y, rcond=None)
    resid = y - X @ beta
    cov = resid @ resid / (len(y) - 2) * np.linalg.inv(X.T @ X)
    t = beta / np.sqrt(np.diag(cov))
    p = [math.erfc(abs(v) / math.sqrt(2)) for v in t]  # two-sided, normal approximation
    return beta, t, p


def article_stats(returns: pd.Series, with_regression: bool = True) -> dict:
    r = returns.dropna()
    s = performance(r)
    out = {
        "Total Return %": s["total_return"] * 100,
        "IRR %": s["annualized_return"] * 100,
        "Vol %": s["annualized_vol"] * 100,
        "Sharpe": s["sharpe"],
        "Hit Ratio %": s["hit_ratio"] * 100,
        "MDD %": s["max_drawdown"] * 100,
    }
    if with_regression:
        b = ibov_ret.reindex(r.index)
        ok = b.notna()
        beta, t, p = ols(r[ok].to_numpy(), b[ok].to_numpy())
        out.update({"Alpha % (yearly)": beta[0] * 252 * 100, "alpha p": p[0], "Beta": beta[1]})
    out.update({"Skew": float(r.skew()), "Worst Day %": r.min() * 100, "Best Day %": r.max() * 100})
    return out


def benchmark_returns(dates) -> pd.Series:
    return ibov_ret.reindex(dates)


def equity_figure(title, curves: dict, height=420):
    fig = go.Figure([go.Scatter(x=c.index, y=c, name=n, mode="lines") for n, c in curves.items()])
    fig.update_layout(title=title, height=height, margin=dict(t=50, b=20), legend_orientation="h",
                      yaxis_title="R$ (log scale)", yaxis_type="log")
    fig.show()


def normalised(res, name=None):
    r = res["ret"].dropna()
    return CAPITAL * (1 + r).cumprod()


def day_picture(date, configs: dict, height_per_row=330):
    rows = len(configs)
    fig = make_subplots(rows=rows, cols=1, shared_xaxes=True, vertical_spacing=0.07,
                        subplot_titles=[f"{name}" for name in configs])
    sel = day == pd.Timestamp(date)
    t = bars.index[sel]
    vw = session_vwap(bars)
    for row, (name, cfg) in enumerate(configs.items(), start=1):
        _, result = run(**{k: v for k, v in cfg.items()})
        config = replace(BASE, **cfg)
        sigma = noise_sigma(bars[["open", "close"]], config.lookback, config.min_days)
        band = noise_bands(bars, sigma, config.band_mult)
        held = strategy_exposure(bars, config)
        close = bars.loc[sel, "close"]
        mark = held[sel] != 0
        fig.add_trace(go.Scatter(x=t, y=close, name="close", line=dict(color="#1f77b4", width=1.4), showlegend=(row == 1)), row=row, col=1)
        fig.add_trace(go.Scatter(x=t, y=band.loc[sel, "ub"], name="upper band", line=dict(color="#2ca02c", dash="dash"), showlegend=(row == 1)), row=row, col=1)
        fig.add_trace(go.Scatter(x=t, y=band.loc[sel, "lb"], name="lower band", line=dict(color="#d62728", dash="dash"), showlegend=(row == 1)), row=row, col=1)
        if config.stop in ("vwap_band", "vwap_only"):
            fig.add_trace(go.Scatter(x=t, y=vw[sel], name="VWAP", line=dict(color="#ff7f0e"), showlegend=(row == 1)), row=row, col=1)
        fig.add_trace(go.Scatter(x=t[mark], y=close[mark], mode="markers", name="position held", showlegend=(row == 1),
                                 marker=dict(size=4, color=np.where(held[sel][mark] > 0, "#2ca02c", "#d62728"))), row=row, col=1)
        net = result.loc[pd.Timestamp(date), "net"]
        ret = result.loc[pd.Timestamp(date), "ret"] * 100
        fig.layout.annotations[row - 1].text = f"{name}: {ret:+.2f}% of the account on the day (R$ {net:+,.0f})"
    fig.update_layout(height=height_per_row * rows + 60, margin=dict(t=60, b=20), legend_orientation="h",
                      title=pd.Timestamp(date).strftime("%A %d %B %Y"))
    fig.show()

## 2 Data

The article: 1-minute OHLCV bars for SPY and the VIX from IQFeed, May 2007 to April 2024. Here: 1-minute bars for `WIN$N`, the continuous series of the WIN mini Ibovespa future, exported from MetaTrader 5 (São Paulo wall-clock time, real volume), development period. The series is **not** adjusted for contract rolls: on each roll Wednesday the overnight gap shows the carry between contracts (+1% to +2.7%). Intraday bars are unaffected; the previous close used by the bands and the volatility target is. The owner's decision is to treat roll days as normal days (`docs/brief.md`).

In [ ]:
last_bar = bars.groupby(day).apply(lambda g: g.index[-1].strftime("%H:%M"))
info = pd.Series(
    {
        "Instrument": "WIN$N (mini Ibovespa future, R$ 0.20 a point)",
        "Source": "MetaTrader 5 export, 1-minute bars, real volume",
        "Period (development)": f"{bars.index[0].date()} to {bars.index[-1].date()}",
        "Trading days": day.nunique(),
        "Bars": f"{len(bars):,}",
        "Bars a day (median)": int(bars.groupby(day).size().median()),
        "Session": "09:00 to 18:25 (the last bar is 18:24); 17:55 in Q2 and Q3 of 2022 and 2023",
        "Benchmark": "IBOV close (price index, no dividends)",
        "Volatility index": "none: 20-day realised volatility of the IBOV (proxy)",
    },
    name="data",
)
display(info.to_frame())
display(pd.DataFrame({"days": bars.groupby(bars.index.year).apply(lambda g: g.index.normalize().nunique()),
                      "bars": bars.groupby(bars.index.year).size()}).T)

In [ ]:
minute = bars.index.hour * 60 + bars.index.minute
profile = bars["volume"].groupby(minute).median()
profile.index = [f"{m // 60:02d}:{m % 60:02d}" for m in profile.index]
fig = go.Figure(go.Scatter(x=profile.index, y=profile, mode="lines"))
fig.update_layout(title="Median volume per minute by time of day (contracts)", height=330, margin=dict(t=50, b=20),
                  xaxis=dict(tickmode="array", tickvals=[f"{h:02d}:00" for h in range(9, 19)]))
fig.show()

## 3 Strategy description

### 3.1 The Noise Area

For each time of day, the average absolute move from the open over the previous 14 days is `sigma`. The Noise Area is the region between `max(open, previous close) * (1 + sigma)` and `min(open, previous close) * (1 - sigma)`. The article says the average move typically grows through the session and peaks at the close. Table below: `sigma` at several times of day, averaged over the development days, for WIN.

In [ ]:
sigma_all = noise_sigma(bars[["open", "close"]])
at_time = sigma_all.groupby(minute).mean() * 100
rows = {f"{h:02d}:{m:02d}": at_time.get(h * 60 + m, np.nan) for h, m in [(9, 30), (10, 0), (10, 30), (12, 0), (14, 0), (15, 30), (17, 0), (18, 0), (18, 24)]}
display(pd.Series(rows, name="mean sigma, % from the open").to_frame().T)
fig = go.Figure(go.Scatter(x=[f"{m // 60:02d}:{m % 60:02d}" for m in at_time.index], y=at_time, mode="lines"))
fig.update_layout(title="Mean sigma by time of day (%): how far from the open counts as noise",
                  height=320, margin=dict(t=50, b=20), xaxis=dict(tickmode="array", tickvals=[f"{h:02d}:00" for h in range(9, 19)]))
fig.show()

**Figure 1** The time-varying boundary area on one development day, with the previous close (the overnight gap widens the band on the side the open moved away from). The day is arbitrary, not chosen for a clean picture.

In [ ]:
def nearest_day(date):
    days = run()[1].index
    return days[days.get_indexer([pd.Timestamp(date)], method="nearest")[0]]


d1 = nearest_day("2024-03-14")
sel = day == d1
bands_all = noise_bands(bars, sigma_all)
t = bars.index[sel]
fig = go.Figure(
    [
        go.Scatter(x=t, y=bands_all.loc[sel, "ub"], name="upper band", line=dict(color="#2ca02c")),
        go.Scatter(x=t, y=bands_all.loc[sel, "lb"], name="lower band", line=dict(color="#d62728"), fill="tonexty", fillcolor="rgba(150,150,150,0.2)"),
        go.Scatter(x=t, y=bars.loc[sel, "close"], name="close", line=dict(color="#1f77b4", width=1.5)),
    ]
)
fig.update_layout(title=f"Figure 1. Noise Area, {d1.date()}", height=380, margin=dict(t=50, b=20), legend_orientation="h")
fig.show()

### 3.2 The base model: the opposite band as stop

*Article:* decisions every 30 minutes; long above the upper band, short below the lower band; positions are closed at the market close, or when the price crosses the **opposite** band, in which case the position reverses. Size: 100% of the equity available at the start of the day (here, whole WIN contracts at a leverage of 1, `contracts = round(equity / (open * R$ 0.20))`). **My reading of the text** (brief, section 10): the last breakout is held until the opposite band is crossed at a decision time or the day ends, with no VWAP.

In [ ]:
base_cfg = dict(stop="opposite_band", leverage=1.0)
cfg_a, res_a = run(**base_cfg)
traded_a = res_a[res_a["ret"].notna()]
dates = traded_a.index
bench = benchmark_returns(dates)
table1 = pd.DataFrame(
    {
        "Momentum, opposite band, 100% size": article_stats(res_a["ret"]),
        "IBOV (buy and hold)": article_stats(bench, with_regression=False),
    }
).T
print("Table 1. Summary statistics of the base model (opposite band as the stop) and the IBOV")
print(f"{len(dates)} days; contracts a day: mean {traded_a['contracts'].mean():.1f}")
display(table1[["Total Return %", "IRR %", "Vol %", "Sharpe", "Hit Ratio %", "MDD %", "Alpha % (yearly)", "alpha p", "Beta"]])

**Figure 2** Two days with a clear trend and the trades of the base model, chosen as the largest winning long and short days of the base model: they show what the model is for and are **not** typical (the article's own days, 31 Jan and 29 Apr 2022, were also chosen to illustrate).

In [ ]:
trades_a = trade_list(bars, cfg_a, res_a)
long_day = trades_a[trades_a["side"] == 1].groupby("day")["points"].sum().idxmax()
short_day = trades_a[trades_a["side"] == -1].groupby("day")["points"].sum().idxmax()
print("best long day:", long_day.date(), "| best short day:", short_day.date())
day_picture(long_day, {"(a) long: base model": base_cfg})
day_picture(short_day, {"(b) short: base model": base_cfg})

**Figure 3** Equity of the base model against the IBOV, both from R$ 100,000 (log scale).

In [ ]:
equity_figure("Figure 3. Base model (opposite band, 100% size) against the IBOV",
              {"Momentum, opposite band": normalised(res_a), "IBOV buy and hold": CAPITAL * (1 + bench.dropna()).cumprod()})

**Figure 4 and 5** The base model's weakness: a day when the market turns and the opposite band is far. Below, the worst day of the base model under the three exits the article builds up: the opposite band (Figure 4), the current band alone, and the current band with the VWAP (Figure 5).

In [ ]:
worst = traded_a["net"].idxmin()
print("worst day of the base model:", worst.date(), f"{traded_a.loc[worst, 'ret'] * 100:.2f}% of the account")
day_picture(
    worst,
    {
        "Opposite band stop (Figure 4)": dict(stop="opposite_band", leverage=1.0),
        "Stop = current band (Figure 5a)": dict(stop="current_band", leverage=1.0),
        "Stop = current band and VWAP (Figure 5b)": dict(stop="vwap_band", leverage=1.0),
    },
)

### 3.3 The VWAP and the current band as the trailing stop (Table 2, Figure 6)

A long is closed when the price falls below `max(upper band, VWAP)`; a short when it rises above `min(lower band, VWAP)`. The size stays at 100% of the equity.

In [ ]:
cfg_b, res_b = run(stop="vwap_band", leverage=1.0)
table2 = pd.DataFrame(
    {
        "Momentum, opposite band": article_stats(res_a["ret"]),
        "Momentum, current band + VWAP": article_stats(res_b["ret"]),
        "IBOV (buy and hold)": article_stats(bench, with_regression=False),
    }
).T
print("Table 2. Opposite band, current band + VWAP, and the IBOV")
display(table2[["Total Return %", "IRR %", "Vol %", "Sharpe", "Hit Ratio %", "MDD %", "Alpha % (yearly)", "alpha p", "Beta"]])
equity_figure("Figure 6. Opposite band, current band + VWAP, and the IBOV",
              {"Opposite band": normalised(res_a), "Current band + VWAP": normalised(res_b), "IBOV": CAPITAL * (1 + bench.dropna()).cumprod()})

### 3.4 Volatility-targeted size (Table 3, Figure 7)

`contracts = round(equity / (open * R$ 0.20) * min(2% / daily volatility, 4))`, the daily volatility being the one of the authors' notebook (the 14 returns before the previous day).

In [ ]:
cfg_c, res_c = run(stop="vwap_band")  # the final model, volatility target
traded_c = res_c[res_c["ret"].notna()]
table3 = pd.DataFrame(
    {
        "Momentum, opposite band": article_stats(res_a["ret"]),
        "Momentum, current band + VWAP": article_stats(res_b["ret"]),
        "Momentum, current band + VWAP, dynamic": article_stats(res_c["ret"]),
        "IBOV (buy and hold)": article_stats(bench, with_regression=False),
    }
).T
print("Table 3. With the volatility-targeted size")
print(f"dynamic size: contracts a day mean {traded_c['contracts'].mean():.1f}, max {traded_c['contracts'].max()}")
display(table3[["Total Return %", "IRR %", "Vol %", "Sharpe", "Hit Ratio %", "MDD %", "Alpha % (yearly)", "alpha p", "Beta"]])
equity_figure("Figure 7. All three models and the IBOV",
              {"Opposite band": normalised(res_a), "Current band + VWAP": normalised(res_b),
               "Current band + VWAP, dynamic size": normalised(res_c), "IBOV": CAPITAL * (1 + bench.dropna()).cumprod()})

**Table 4** Trade-level statistics. A trade is a run of the same position inside a day (a reversal is two trades). *Avg. PnL × contract* is in index points per contract; the article uses dollars per share.

In [ ]:
def trade_table(name, cfg, res):
    tr = trade_list(bars, cfg, res)
    traded_days = int(res["ret"].notna().sum())
    worst_t, best_t = tr.loc[tr["ret"].idxmin()], tr.loc[tr["ret"].idxmax()]
    return pd.Series(
        {
            "Trades": len(tr),
            "Avg. trades a day": len(tr) / traded_days,
            "Hit Ratio % (trades)": (tr["points"] > 0).mean() * 100,
            "Avg. PnL x contract (points)": tr["points"].mean(),
            "Max loss trade %": worst_t["ret"] * 100,
            "  on": worst_t["day"].date(),
            "Max gain trade %": best_t["ret"] * 100,
            "  on ": best_t["day"].date(),
        },
        name=name,
    )


table4 = pd.DataFrame([trade_table("Opposite band", cfg_a, res_a), trade_table("Current band + VWAP", cfg_b, res_b),
                       trade_table("Current band + VWAP, dynamic", cfg_c, res_c)])
print("Table 4. Trade-level statistics")
display(table4)

**Table A1** (appendix) Skewness, worst and best day of the three models and the benchmark.

In [ ]:
print("Table A1. Skewness, worst day and best day")
display(table3[["Skew", "Worst Day %", "Best Day %"]])

## 4 Further investigations

Unless said otherwise, the model is the final one (current band and VWAP, volatility-targeted size), as in the article.

### 4.1 Market volatility regimes (Figure 8)

The article classifies days by the **VIX at the open** and shows the Sharpe ratio of days above each threshold. The VIX does not exist for B3 here: the **proxy** is the 20-day realised volatility of the IBOV (annualised, %), known at the open from the days before.

In [ ]:
r_c = res_c["ret"].dropna()
regime = rv20.reindex(r_c.index)
print(f"proxy: min {regime.min():.1f}, median {regime.median():.1f}, max {regime.max():.1f} (% a year)")
rows = []
for threshold in (0, 10, 12, 14, 16, 18, 20, 25):
    sel = r_c[regime >= threshold]
    rows.append({"proxy >=": threshold, "days": len(sel), "Sharpe": sel.mean() / sel.std() * math.sqrt(252) if len(sel) > 30 else np.nan})
vix_table = pd.DataFrame(rows).set_index("proxy >=")
display(vix_table.T)
fig = go.Figure(go.Bar(x=[f">= {t}" for t in vix_table.index], y=vix_table["Sharpe"], text=vix_table["days"].astype(str) + " days", textposition="outside"))
fig.update_layout(title="Figure 8. Sharpe ratio by level of the volatility proxy at the open (days with fewer than 30 are blank)",
                  height=340, margin=dict(t=60, b=20))
fig.show()

### 4.2 Daily patterns (Table 5)

The article tests whether the strategy's profit changes after eight well-known daily patterns, identified on the previous day. The definitions below follow its text; the daily bars are WIN's own (session open to close).

In [ ]:
d = bars.groupby(day).agg(open=("open", "first"), high=("high", "max"), low=("low", "min"), close=("close", "last"))
rng = d["high"] - d["low"]
pos = lambda x: (x - d["low"]) / rng  # where a price sits in the day's range, 0 to 1
prev_high, prev_low = d["high"].shift(1), d["low"].shift(1)
patterns = {
    "NR4": rng == rng.rolling(4).min(),
    "NR7": rng == rng.rolling(7).min(),
    "ID (inside day)": (d["high"] < prev_high) & (d["low"] > prev_low),
    "OD (outside day)": (d["high"] > prev_high) & (d["low"] < prev_low),
    "Triangle": (d["high"] < d["high"].shift(1)) & (d["high"] < d["high"].shift(2)) & (d["low"] > d["low"].shift(1)) & (d["low"] > d["low"].shift(2)),
    "Trend day": (((pos(d["open"]) < 0.15) & (pos(d["close"]) > 0.85)) | ((pos(d["open"]) > 0.85) & (pos(d["close"]) < 0.15))) & (rng > rng.rolling(14).mean().shift(1)),
    "Big tail": ((pos(d["open"]) > 0.75) & (pos(d["close"]) > 0.75)) | ((pos(d["open"]) < 0.25) & (pos(d["close"]) < 0.25)),
    "Strong/weak closure": (pos(d["close"]) > 0.9) | (pos(d["close"]) < 0.1),
}


def hit(r):
    # winners among the days that traded (a day with no trade is not a miss), as `performance`
    return (r > 0).sum() / (r.abs() > 0).sum() * 100


def conditional(name, flag):
    r = r_c[flag.shift(1).reindex(r_c.index).fillna(False).astype(bool)]
    if len(r) < 5:
        return {"Observations": len(r)}
    return {"Observations": len(r), "Avg. PnL (bps)": r.mean() * 1e4, "t-stat": r.mean() / (r.std() / math.sqrt(len(r))),
            "Hit Ratio %": hit(r), "Sharpe": r.mean() / r.std() * math.sqrt(252)}


rows = {"Unconditional": conditional("all", pd.Series(True, index=d.index))}
rows.update({name: conditional(name, flag) for name, flag in patterns.items()})
print("Table 5. The strategy after each daily pattern (identified on the previous day)")
display(pd.DataFrame(rows).T)

### 4.3 Day-of-the-week effect (Table 6)

In [ ]:
rows = {"Unconditional": conditional("all", pd.Series(True, index=d.index))}
names = ["Monday", "Tuesday", "Wednesday", "Thursday", "Friday"]
for i, name in enumerate(names):
    r = r_c[r_c.index.dayofweek == i]
    rows[name] = {"Observations": len(r), "Avg. PnL (bps)": r.mean() * 1e4, "t-stat": r.mean() / (r.std() / math.sqrt(len(r))),
                  "Hit Ratio %": hit(r), "Sharpe": r.mean() / r.std() * math.sqrt(252)}
print("Table 6. The strategy by day of the week")
display(pd.DataFrame(rows).T)

### 4.4 Volatility multiplier (Figure 9)

`UB = max(open, previous close) * (1 + VM * sigma)`; the article used VM = 1 and found the best Sharpe near 1.5 in hindsight. Here the same sweep.

In [ ]:
rows = {}
for vm in (0.5, 0.75, 1.0, 1.25, 1.5, 1.75, 2.0):
    _, res_vm = run(stop="vwap_band", band_mult=vm)
    s = performance(res_vm["ret"])
    rows[vm] = {"Total Return %": s["total_return"] * 100, "Sharpe": s["sharpe"], "MDD %": s["max_drawdown"] * 100}
vm_table = pd.DataFrame(rows).T
display(vm_table.T)
fig = make_subplots(rows=2, cols=1, shared_xaxes=True, vertical_spacing=0.1, subplot_titles=("Sharpe ratio", "Total return (%)"))
fig.add_trace(go.Bar(x=[str(v) for v in vm_table.index], y=vm_table["Sharpe"], showlegend=False), row=1, col=1)
fig.add_trace(go.Bar(x=[str(v) for v in vm_table.index], y=vm_table["Total Return %"], showlegend=False), row=2, col=1)
fig.update_layout(title="Figure 9. Effect of the volatility multiplier", height=460, margin=dict(t=60, b=20))
fig.show()

### 4.5 Gamma imbalance (regression on the 5-day RSI)

The article regresses the strategy's daily return (bps) on the 5-day RSI of SPY at the previous close and finds a negative slope (a higher RSI, lower profit). Here the RSI is that of the **IBOV** (Wilder's, 5 days), the nearest available stand-in for the underlying index. The p-value uses the normal approximation.

In [ ]:
ib_rsi = pd.Series(rsi(ibov.to_numpy(dtype=float), 5), index=ibov.index).shift(1)  # at the previous close
x = ib_rsi.reindex(r_c.index)
ok = x.notna()
beta, t, p = ols((r_c[ok] * 1e4).to_numpy(), x[ok].to_numpy())
display(pd.DataFrame({"coefficient": beta, "t-stat": t, "p-value": p}, index=["alpha (bps)", "beta (bps per RSI point)"]))
print(f"{int(ok.sum())} days. The article's beta is -3.25 (p = 0.001), on SPY, 2007 to 2024.")

### 4.6 The impact of commissions and slippage (Figure 10)

Total return as the cost per contract and side grows. R$ 0.25 is the B3 fee; one WIN tick is R$ 1.00, so R$ 1.25 is the fee plus one tick of slippage.

In [ ]:
rows = {}
for fee in (0.0, 0.25, 0.5, 1.0, 1.25, 2.25, 3.25):
    _, res_f = run(stop="vwap_band", fee_per_side=fee)
    s = performance(res_f["ret"])
    rows[f"R$ {fee:.2f}"] = {"Total Return %": s["total_return"] * 100, "Sharpe": s["sharpe"]}
fee_table = pd.DataFrame(rows).T
display(fee_table.T)
fig = go.Figure(go.Bar(x=fee_table.index, y=fee_table["Total Return %"]))
fig.update_layout(title="Figure 10. Total return against the cost per contract and side", height=330, margin=dict(t=50, b=20))
fig.show()

## 5 Conclusion

The article's headline is a 19.6% yearly return and a Sharpe ratio of 1.33 for the final model on SPY, 2007 to 2024. The same rules, with the same parameters and on 1-minute WIN bars, give the following on the development period. Read it with the cautions at the top: three years, one market, and no costs beyond the B3 fee.

In [ ]:
paper = {"Total Return %": 1985, "IRR %": 19.6, "Vol %": 14.3, "Sharpe": 1.33, "Hit Ratio %": 43, "MDD %": 25}
ours = article_stats(res_c["ret"], with_regression=False)
display(pd.DataFrame({"Paper: SPY, 2007 to 2024": paper, "Here: WIN, development period": {k: ours[k] for k in paper}}).T)
print(f"t-statistic of the mean daily return here: {performance(res_c['ret'])['t_stat']:.2f}; days: {int(performance(res_c['ret'])['days'])}.")

What the rebuild shows, in short (the numbers are in the tables above and move if the parameters do):

- **The structure of the model is reproducible**: bands, VWAP and volatility targeting run as the authors' notebook runs them (checked against its own loop). What does not carry over automatically is the size of the profit.
- **Each refinement of the article changes the picture** (opposite band, then the VWAP stop, then the volatility target), but on three years and about 750 days the differences between rows are of the order of the noise.
- **Costs and the exit decide the result on WIN**: see section 4.6 and `winfut_intraday_momentum.ipynb`. One tick of slippage per side removes most of the profit.
- **Nothing here replaces the final test.** The owner approves the pass rule and the variants first; the development tables above are for understanding, not for choosing.

## Frequently Asked Questions

The article's own questions that are about numbers, answered on WIN. Not answered because they are not about numbers: Q1 to Q3 (code, data vendors, TradingView), Q9 to Q11, Q16, Q17, Q25. Q15 (the I-Star market impact model) is replaced by section 4.6, a plain cost sweep, because the model's coefficients are calibrated on US stocks. Q24 (performance since publication) is, for this project, the final test and has not been read.

**Q4. Monthly performance table.** Percent of the account, volatility-targeted final model.

In [ ]:
r = r_c
monthly = ((1 + r).groupby([r.index.year, r.index.month]).prod() - 1) * 100
table = monthly.unstack(1)
table.columns = ["Jan", "Feb", "Mar", "Apr", "May", "Jun", "Jul", "Aug", "Sep", "Oct", "Nov", "Dec"][: table.shape[1]]
table["Yearly"] = ((1 + r).groupby(r.index.year).prod() - 1) * 100
table.index.name = "Year"
display(table.round(1))

**Q5. Do the long and short legs contribute equally?** Equity of the long-only, short-only and combined strategies (final model).

In [ ]:
_, res_long = run(stop="vwap_band", sides="long")
_, res_short = run(stop="vwap_band", sides="short")
equity_figure("Q5. Long leg, short leg and both", {"Long only": normalised(res_long), "Short only": normalised(res_short), "Long and short": normalised(res_c)})
display(pd.DataFrame({"Long only": article_stats(res_long["ret"], False), "Short only": article_stats(res_short["ret"], False),
                      "Long and short": article_stats(res_c["ret"], False)}).T[["Total Return %", "IRR %", "Vol %", "Sharpe", "MDD %"]])

**Q6. How does the lookback period for sigma affect the results?** The window is the number of days; at least one less than that must be present (as the notebook's 14 with 13).

In [ ]:
rows = {}
for lookback in (5, 10, 14, 20, 25, 30, 60, 90):
    _, res_l = run(stop="vwap_band", lookback=lookback, min_days=max(1, lookback - 1))
    s = performance(res_l["ret"])
    rows[lookback] = {"Total Return %": s["total_return"] * 100, "IRR %": s["annualized_return"] * 100, "Vol %": s["annualized_vol"] * 100,
                      "Sharpe": s["sharpe"], "MDD %": s["max_drawdown"] * 100}
lookback_table = pd.DataFrame(rows).T.rename_axis("Lookback (days)")
display(lookback_table)

**Q7. The worst quarters.** The article lists the ten worst quarters for the S&P 500. The development window has few full quarters, so all of them are listed, worst IBOV quarter first.

In [ ]:
quarters = r_c.groupby(r_c.index.to_period("Q"))
strategy_q = quarters.apply(lambda x: ((1 + x).prod() - 1) * 100)
counts = quarters.size()
ibov_q = ibov.groupby(ibov.index.to_period("Q")).last().pct_change() * 100  # quarter end to quarter end
full = counts[counts >= 55].index  # quarters with most of their trading days
display(pd.DataFrame({"IBOV %": ibov_q[full], "Strategy %": strategy_q[full], "days": counts[full]}).sort_values("IBOV %"))

**Q8. Is it a hedge for a passive long exposure?** Correlation and beta with the IBOV, and the strategy's average daily return by IBOV daily-return quintile.

In [ ]:
b = ibov_ret.reindex(r_c.index)
beta, t, p = ols(r_c[b.notna()].to_numpy(), b[b.notna()].to_numpy())
print(f"correlation with the IBOV: {np.corrcoef(r_c[b.notna()], b[b.notna()])[0, 1]:+.2f}; beta {beta[1]:+.3f} (p {p[1]:.2f})")
quintile = pd.qcut(b, 5, labels=["worst fifth", "2", "3", "4", "best fifth"])
display(pd.DataFrame({"strategy mean daily return, bps": (r_c * 1e4).groupby(quintile, observed=True).mean(),
                      "IBOV mean daily return, bps": (b * 1e4).groupby(quintile, observed=True).mean()}).T)

**Q12 and Q13. Other markets and futures.** The same final model on the other B3 futures in `samples/` (development period). Per-instrument cost, point value and margin are not known, so this is **gross, with fractional-sized exposure** (volatility target on a very large account, no fee, no margin cap): a comparison of the signal, not a tradable result. Each session ends at its own last bar. BIT has only 8 months of development data. GLD starts in the final period and is not used.

**Be careful with this table.** It is *gross of every cost*, on the same three years for every instrument (so the results are correlated), and BGI and CCM trade only about 190 and 335 minutes a day, so many bars are far apart in time and a cost of even one tick is unknown but probably large against their moves. A Sharpe ratio near 1.5 here is not evidence of a tradable edge; the article's own number for 33 futures over 17 years is an average of 0.60.

In [ ]:
instruments = {"WIN$N": "Equity index", "WSP$N": "Equity index (S&P 500)", "WDO$N": "Currency (USD/BRL)",
               "BGI$N": "Commodity (live cattle)", "CCM$N": "Commodity (corn)", "BIT$N": "Crypto (Bitcoin), from 2024-04"}
unit = IntradayConfig(exit_minute=24 * 60, capital=1e9, point_value=1.0, fee_per_side=0.0, margin_per_contract=0.0)
rows, curves, daily_returns = {}, {}, {}
for symbol, kind in instruments.items():
    b_ = bars if symbol == "WIN$N" else load_bars("../samples", symbol=symbol)
    assert b_.index.max() < pd.Timestamp(FINAL_START.date())
    res_i = run_intraday_momentum(b_, unit)
    ri = res_i["ret"].dropna()
    s = performance(ri)
    rows[symbol] = {"Class": kind, "Days": int(s["days"]), "Total Return %": s["total_return"] * 100, "CAGR %": s["annualized_return"] * 100,
                    "Vol %": s["annualized_vol"] * 100, "Sharpe": s["sharpe"], "MDD %": s["max_drawdown"] * 100,
                    "Hit Ratio %": s["hit_ratio"] * 100, "Skew": float(ri.skew())}
    curves[symbol] = (1 + ri).cumprod()
    daily_returns[symbol] = ri
display(pd.DataFrame(rows).T)
panel = pd.DataFrame({k: v for k, v in daily_returns.items() if k != "BIT$N"}).dropna(how="all")
portfolio = panel.mean(axis=1)
sp = performance(portfolio)
print(f"equal-weight portfolio of the {panel.shape[1]} instruments with the full period: Sharpe {sp['sharpe']:.2f}, total return {sp['total_return'] * 100:.1f}%, max drawdown {sp['max_drawdown'] * 100:.1f}% "
      f"(the article reports 1.65 for 33 futures over 17 years)")
fig = go.Figure([go.Scatter(x=c.index, y=c, name=n, mode="lines") for n, c in curves.items()]
                + [go.Scatter(x=portfolio.index, y=(1 + portfolio).cumprod(), name="equal-weight portfolio", line=dict(color="black", width=3))])
fig.update_layout(title="Q12/Q13. Growth of 1 on each instrument (gross) and an equal-weight portfolio", height=420,
                  margin=dict(t=50, b=20), legend_orientation="h", yaxis_type="log")
fig.show()

**Q14. Is it over-optimised?** The article's answer: few parameters, and the paper's choices are not the best in hindsight. Here the same check from Q6 and section 4.4: where do the article's choices (14 days, a multiplier of 1) stand among the values tried?

In [ ]:
print(f"lookback: the article's 14 days gives a Sharpe of {lookback_table.loc[14, 'Sharpe']:.2f}; the best of those tried is {lookback_table['Sharpe'].idxmax()} days with {lookback_table['Sharpe'].max():.2f}; the worst is {lookback_table['Sharpe'].idxmin()} days with {lookback_table['Sharpe'].min():.2f}.")
print(f"multiplier: the article's 1.0 gives {vm_table.loc[1.0, 'Sharpe']:.2f}; the best is {vm_table['Sharpe'].idxmax()} with {vm_table['Sharpe'].max():.2f}; the worst is {vm_table['Sharpe'].idxmin()} with {vm_table['Sharpe'].min():.2f}.")
print("Neither choice was tuned on this data, but the spread across values is the size of the noise: the ranking should not be read as a result.")

**Q18. Intraday seasonality.** Average profit per day by time of day, in index points per contract, for the final model held to the close.

In [ ]:
exposure = strategy_exposure(bars, cfg_c)
step = bars["close"].groupby(day).diff()
points = (exposure * step).fillna(0.0)
bucket = pd.Series((minute // 30) * 30, index=bars.index)
per_bucket = points.groupby(bucket).sum() / day.nunique()
labels = [f"{m // 60:02d}:{m % 60:02d}" for m in per_bucket.index]
fig = make_subplots(rows=2, cols=1, shared_xaxes=True, vertical_spacing=0.1, subplot_titles=("Average points per contract and day, by 30-minute bucket", "Cumulative through the day"))
fig.add_trace(go.Bar(x=labels, y=per_bucket, showlegend=False), row=1, col=1)
fig.add_trace(go.Scatter(x=labels, y=per_bucket.cumsum(), mode="lines", showlegend=False), row=2, col=1)
fig.update_layout(title="Q18. Where in the day the model earns", height=480, margin=dict(t=70, b=20))
fig.show()

**Q19 and Q20. Short trades and the volatility level.** The article regresses each short trade's profit (bps) on the day's VIX and then totals short trades above VIX thresholds. With the **proxy** (20-day realised volatility of the IBOV, from the days before).

In [ ]:
trades_c = trade_list(bars, cfg_c, res_c)
shorts = trades_c[trades_c["side"] == -1].copy()
shorts["bps"] = shorts["points"] / shorts["entry_price"] * 1e4  # a short's trade return, bps of the entry price
shorts["proxy"] = rv20.reindex(shorts["day"]).to_numpy()
ok = shorts["proxy"].notna()
beta, t, p = ols(shorts.loc[ok, "bps"].to_numpy(), shorts.loc[ok, "proxy"].to_numpy())
print(f"Q19: {int(ok.sum())} short trades; slope {beta[1]:+.2f} bps per point of the proxy, p = {p[1]:.2f} (the article: no relationship, p = 0.91)")
rows = {}
for threshold in (0, 10, 12, 14, 16, 18, 20):
    sel = shorts[ok & (shorts["proxy"] >= threshold)]
    rows[f">= {threshold}"] = {"short trades": len(sel), "total return of the short trades %": ((1 + sel["bps"] / 1e4).prod() - 1) * 100}
q20 = pd.DataFrame(rows).T
display(q20.T)
fig = go.Figure(go.Bar(x=q20.index, y=q20["total return of the short trades %"], text=q20["short trades"].astype(str) + " trades", textposition="outside"))
fig.update_layout(title="Q20. Total return of short trades when the proxy is above a level", height=330, margin=dict(t=50, b=20))
fig.show()

**Q21. Should shorts be opened only below key moving averages?** Total return (unlevered, trade by trade) of the short trades when the IBOV was below its 100, 150 or 200-day average at the previous close, against the ones above it and all of them.

In [ ]:
rows = {}
def compounded(frame):
    return ((1 + frame["bps"] / 1e4).prod() - 1) * 100 if len(frame) else np.nan

for n in (100, 150, 200):
    sma = ibov.rolling(n).mean()
    below = (ibov < sma).shift(1)  # known at the previous close
    flag = below.reindex(shorts["day"]).to_numpy()
    flag = pd.Series(flag, index=shorts.index).fillna(False).astype(bool)
    rows[f"SMA {n}"] = {"all short trades %": compounded(shorts), "only below the SMA %": compounded(shorts[flag]),
                        "only above the SMA %": compounded(shorts[~flag]), "trades below": int(flag.sum()), "trades above": int((~flag).sum())}
display(pd.DataFrame(rows).T)

**Q22. A VWAP-only trailing stop.** The article compares the final model with one that uses the VWAP alone as the stop (entries on a band breach, my reading in the brief). Its table is for 1x leverage and the dynamic size.

In [ ]:
cfg_v1, res_v1 = run(stop="vwap_only", leverage=1.0)
cfg_vd, res_vd = run(stop="vwap_only")
rows = {}
for name, (cfg_, res_) in {
    "Opposite band, 1x": (cfg_a, res_a),
    "Current band + VWAP, 1x": (cfg_b, res_b),
    "Current band + VWAP, dynamic": (cfg_c, res_c),
    "VWAP only, 1x": (cfg_v1, res_v1),
    "VWAP only, dynamic": (cfg_vd, res_vd),
}.items():
    s = performance(res_["ret"])
    rows[name] = {"Trades": len(trade_list(bars, cfg_, res_)), "Total Return %": s["total_return"] * 100, "IRR %": s["annualized_return"] * 100,
                  "Vol %": s["annualized_vol"] * 100, "Sharpe": s["sharpe"], "Hit Ratio %": s["hit_ratio"] * 100, "MDD %": s["max_drawdown"] * 100}
display(pd.DataFrame(rows).T)

**Q23. How does the average trade profit change over the years?** Average profit per trade in index points per contract (the article: dollars per share), final model.

In [ ]:
by_year = trades_c.groupby(trades_c["day"].dt.year).agg(trades=("points", "size"), avg_points=("points", "mean"), avg_reais_per_contract=("points", lambda x: x.mean() * 0.2))
display(by_year.T)
fig = go.Figure(go.Bar(x=by_year.index.astype(str), y=by_year["avg_points"]))
fig.update_layout(title="Q23. Average profit per trade, index points per contract", height=320, margin=dict(t=50, b=20))
fig.show()

---
*The final test period (2025-01-02 to 2026-10-30) has not been read by this notebook. Question 24 of the article, the performance after publication, is that period for this project.*